# NASN Task 03B — MARBERT Baseline v1.0 on Colab GPU

This notebook is **unexecuted preparation**, not training evidence. Choose **Runtime → Change runtime type → GPU** first. It clones baseline commit `c8dc698780cb06c8838c557f159bdaed7507aeda`, preserves its CLI and protocol, and writes experiments directly to your mounted Google Drive. No official test evaluation, split reconstruction or random-encoder fallback.

Run cells in order. Stop on any failure. A completed run requires `run_status.status == completed`, checkpoint hashes, and verified validation reload. An interrupted run is incomplete; the baseline does not implement resumable optimizer state. Start a fresh seeded run after changing only microbatch/accumulation for an OOM.


In [ ]:
from pathlib import Path
import datetime, hashlib, importlib.util, json, os, shutil, subprocess, sys, uuid
REFERENCE = "c8dc698780cb06c8838c557f159bdaed7507aeda"
REPOSITORY = "https://github.com/haneenbiate/-.git"
REPO = Path("/content/nasn-task03b-baseline")
CACHE = Path("/content/nasn-task03b-cache")
RUN_ID = "marbert-v1-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
# Optional verified local snapshot directory in Drive (never a trained classifier):
LOCAL_SNAPSHOT = None  # e.g. "/content/drive/MyDrive/NASN/pretrained/MARBERT-88e1fa"
MICRO_BATCH_SIZE = None  # auto conservative VRAM heuristic, or explicitly 4, 2, 1
CUDA_INDEX = "https://download.pytorch.org/whl/cu126"
def call(args, **kwargs):
    return subprocess.run([str(a) for a in args], check=True, **kwargs)
if not REPO.exists():
    call(["git","clone","--no-checkout",REPOSITORY,REPO],env={**os.environ,"GIT_TERMINAL_PROMPT":"0"})
    call(["git","-C",REPO,"checkout","--detach",REFERENCE])
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip() == REFERENCE
assert not subprocess.check_output(["git","-C",str(REPO),"status","--porcelain","--untracked-files=no"],text=True).strip(), "Pinned tracked files were modified"
print("Pinned baseline checkout:", REFERENCE)


## Durable output gate

Google Drive authorization happens interactively in Colab. Ensure at least 10 GB of Drive quota and local disk space. Each run gets a new directory; previous Task 03 evidence is untouched. Checkpoints are written to Drive during training, not copied there only at the end. Drive can still disconnect or exhaust quota; do not claim durability until final readback/hash verification passes.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Drive must be mounted; do not use an ordinary temporary directory"
RUN_ROOT = Path("/content/drive/MyDrive/NASN/task03b") / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
probe = RUN_ROOT / ".storage_probe"
probe.write_bytes(b"NASN durable storage write/read probe")
assert probe.read_bytes() == b"NASN durable storage write/read probe"
probe.unlink()
assert shutil.disk_usage("/content").free >= 10 * 1024**3, "Insufficient local disk"
EXPERIMENT = RUN_ROOT / "experiment"  # leave absent/empty for original CLI
print("Durable run directory:", RUN_ROOT)


## Exact interpreter and isolated dependencies

Task 02 enforces Python **3.12.14**, not merely 3.12. Use a managed, isolated interpreter and venv so Colab's global libraries remain untouched. All TLS/artifact verification stays enabled. The CUDA 12.6 PyTorch 2.8.0 build must pass an actual CUDA kernel test on the assigned driver; wheel availability alone is not GPU compatibility evidence.


In [ ]:
BOOTSTRAP = Path("/content/nasn-bootstrap")
call([sys.executable,"-m","pip","install","--no-cache-dir","--target",BOOTSTRAP,"uv==0.13.0"])
UV = BOOTSTRAP / "bin/uv"
BOOT_ENV = {**os.environ,"UV_PYTHON_INSTALL_DIR":"/content/nasn-python","UV_CACHE_DIR":"/content/nasn-uv-cache","UV_PYTHON_BIN_DIR":"/content/nasn-python-bin"}
call([UV,"python","install","--system-certs","3.12.14"],env=BOOT_ENV)
MANAGED_PYTHON = subprocess.check_output([str(UV),"python","find","--managed-python","3.12.14"],env=BOOT_ENV,text=True).strip()
VENV = Path("/content/nasn-marbert-venv")
assert not VENV.exists(), "Use a fresh runtime rather than overwriting an existing venv"
call([MANAGED_PYTHON,"-m","venv",VENV])
PYTHON = VENV / "bin/python"
assert subprocess.check_output([str(PYTHON),"-c","import platform; print(platform.python_version())"],text=True).strip() == "3.12.14"
call([PYTHON,"-m","pip","install","--no-cache-dir","torch==2.8.0+cu126","--index-url",CUDA_INDEX])
call([PYTHON,"-m","pip","install","--no-cache-dir","-r",REPO / "requirements-marbert.txt"])
check = call([PYTHON,"-m","pip","check"],capture_output=True,text=True)
(RUN_ROOT / "dependency_check.txt").write_text(check.stdout + check.stderr)
(RUN_ROOT / "dependency_freeze.txt").write_text(subprocess.check_output([str(PYTHON),"-m","pip","freeze"],text=True))
ENV = {**os.environ,"PYTHONPATH":str(REPO),"HF_HOME":str(CACHE),"HF_HUB_DISABLE_XET":"1","CUBLAS_WORKSPACE_CONFIG":":4096:8"}


## Embedded integrity helper and official snapshot lock

These preparation/storage helpers are materialized **outside** the baseline checkout. They contain no training algorithm. The weight hash comes from official Hub LFS metadata at the exact model revision, not a checksum invented from a local file. The four tokenizer/config files also have verified Git blob identities. The embedded helper bytes and lock are saved with the run for inspection.


In [ ]:
HELPER_SOURCE = '"""External-runtime integrity and storage helpers; no training algorithm."""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nREFERENCE = \'c8dc698780cb06c8838c557f159bdaed7507aeda\'\nMODEL = \'UBC-NLP/MARBERT\'\nREVISION = \'88e1fa192dd723cf0b3563500aec46209762eb22\'\nREQUIRED_FILES = {\'config.json\', \'tokenizer_config.json\', \'special_tokens_map.json\', \'vocab.txt\', \'pytorch_model.bin\'}\nWEIGHTS_SHA256 = \'801862ace5fc9e6b135b19476dbda5c7de1e8673b5b9d55db08d79e2c30b9dca\'\nREPORT_FILES = {\'training_config.json\',\'resolved_training_config.json\',\'preflight.json\',\'environment.json\',\n    \'run_status.json\',\'methodology.md\',\'epoch_history.csv\',\'validation_metrics.json\',\'per_class_results.csv\',\n    \'confusion_matrix.csv\',\'validation_predictions.csv\',\'subgroup_metrics.json\',\'checkpoint_hashes.json\',\n    \'model_loading_info.json\',\'near_duplicate_audit.csv\',\'near_duplicate_summary.csv\',\'manual_review_queue.csv\'}\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    result = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef save(path, value):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(value, indent=2) + \'\\n\')\n\n\ndef safe_error(error):\n    return re.sub(r\'\\?[^\\s\\\'"<>)]*\', \'?[query omitted]\', str(error))\n\n\ndef validate_lock(lock):\n    require(lock[\'repo_id\'] == MODEL and lock[\'revision\'] == REVISION, \'Snapshot revision/identity differs from baseline\')\n    require(set(lock[\'files\']) == REQUIRED_FILES, \'Snapshot lock must cover all required official files\')\n    require(lock[\'files\'][\'pytorch_model.bin\'][\'sha256\'] == WEIGHTS_SHA256, \'Untrusted pretrained weight hash\')\n    for name, details in lock[\'files\'].items():\n        require(Path(name).name == name, \'Unsafe snapshot filename\')\n        require(bool(re.fullmatch(\'[0-9a-f]{64}\', details[\'sha256\'])), \'Invalid snapshot checksum\')\n\n\ndef verify_snapshot(snapshot, lock):\n    validate_lock(lock)\n    snapshot = Path(snapshot)\n    require(snapshot.is_dir(), \'Snapshot directory missing\')\n    extra = {p.name for p in snapshot.iterdir()} - REQUIRED_FILES - {\'snapshot_revision.json\'}\n    require(not extra, f\'Unexpected snapshot files (could shadow verified weights/tokenizer): {sorted(extra)}\')\n    hashes = {}\n    for name, expected in lock[\'files\'].items():\n        path = snapshot / name\n        require(path.is_file(), f\'Official snapshot file missing: {name}\')\n        require(path.stat().st_size == expected[\'size\'], f\'Snapshot size mismatch: {name}\')\n        hashes[name] = sha256(path)\n        require(hashes[name] == expected[\'sha256\'], f\'Snapshot SHA-256 mismatch: {name}\')\n    return hashes\n\n\ndef cache_snapshot(source, cache, lock):\n    """Stage verified bytes in the standard exact-revision Hub cache; no source edits."""\n    hashes = verify_snapshot(source, lock)\n    target = Path(cache) / \'models--UBC-NLP--MARBERT\' / \'snapshots\' / REVISION\n    target.mkdir(parents=True, exist_ok=True)\n    extra = {p.name for p in target.iterdir()} - REQUIRED_FILES - {\'snapshot_revision.json\'}\n    require(not extra, f\'Unverified files in exact-revision cache: {sorted(extra)}\')\n    for name, digest in hashes.items():\n        destination = target / name\n        if destination.exists():\n            require(sha256(destination) == digest, f\'Conflicting existing Hub cache file: {name}\')\n        else:\n            shutil.copyfile(Path(source) / name, destination)\n    verify_snapshot(target, lock)\n    return target\n\n\ndef acquire_snapshot(cache, lock, report, local=None):\n    validate_lock(lock)\n    try:\n        if local:\n            # Local revision declaration is checked, but byte trust comes from the external official lock.\n            declaration = json.loads((Path(local) / \'snapshot_revision.json\').read_text())\n            require(declaration[\'repo_id\'] == MODEL and declaration[\'revision\'] == REVISION,\n                    \'Local snapshot revision declaration mismatch\')\n            snapshot = cache_snapshot(local, cache, lock)\n            origin = \'verified local snapshot\'\n        else:\n            os.environ[\'HF_HUB_DISABLE_XET\'] = \'1\'\n            from huggingface_hub import snapshot_download\n            from huggingface_hub.utils import logging\n            logging.set_verbosity_error()\n            snapshot = Path(snapshot_download(MODEL, revision=REVISION, cache_dir=cache,\n                allow_patterns=sorted(REQUIRED_FILES), max_workers=1))\n            origin = \'official Hub exact revision\'\n        hashes = verify_snapshot(snapshot, lock)\n        save(report, {\'status\':\'verified\',\'repo_id\':MODEL,\'revision\':REVISION,\'source\':origin,\n                      \'snapshot\':str(snapshot),\'files_sha256\':hashes,\'encoder_load_verified\':False})\n        return snapshot\n    except Exception as exc:\n        save(report, {\'status\':\'blocked_or_invalid\',\'repo_id\':MODEL,\'revision\':REVISION,\n                      \'error\':safe_error(exc),\'weights_verified\':False,\'encoder_load_verified\':False})\n        raise ValueError(safe_error(exc)) from None\n\n\ndef gpu_report(path):\n    import torch\n    info = {\'torch\':torch.__version__,\'torch_cuda_build\':torch.version.cuda,\n            \'python\':sys.version,\'python_executable\':sys.executable,\n            \'cuda_available\':torch.cuda.is_available(),\'reference_commit\':REFERENCE}\n    try:\n        require(torch.version.cuda is not None and torch.cuda.is_available(), \'CUDA GPU unavailable; no CPU fallback\')\n        free, total = torch.cuda.mem_get_info()\n        info.update(gpu=torch.cuda.get_device_name(0), capability=list(torch.cuda.get_device_capability(0)),\n                    free_vram_bytes=free,total_vram_bytes=total)\n        info[\'nvidia_smi\'] = subprocess.check_output([\'nvidia-smi\',\'--query-gpu=name,driver_version,memory.total,memory.free\',\n            \'--format=csv,noheader\'], text=True).strip()\n        require(free >= 6 * 1024**3, \'Less than 6 GiB free VRAM; request a larger GPU runtime\')\n        value = torch.ones((8,8),device=\'cuda\')\n        require(torch.isfinite(value @ value).all().item(), \'CUDA kernel compatibility test failed\')\n        info[\'status\'] = \'gpu_gate_passed\'\n        save(path, info)\n    except Exception as exc:\n        info.update(status=\'blocked\',error=safe_error(exc))\n        save(path, info)\n        raise\n\n\ndef make_config(original, gpu, cache, micro=None):\n    require(gpu[\'status\'] == \'gpu_gate_passed\', \'GPU gate has not passed\')\n    if micro is None:\n        free = gpu[\'free_vram_bytes\'] / 1024**3\n        micro = 4 if free >= 12 else 2 if free >= 8 else 1\n    require(type(micro) is int and micro in [1,2,4], \'Allowed microbatch sizes are 4, 2, 1\')\n    require(original[\'model_revision\'] == REVISION and original[\'batch_size\'] == 16, \'Unexpected baseline configuration\')\n    config = {**original,\'device\':\'cuda\',\'micro_batch_size\':micro,\'gradient_accumulation_steps\':16 // micro,\n              \'cache_directory\':str(cache)}\n    allowed = {\'device\',\'micro_batch_size\',\'gradient_accumulation_steps\',\'cache_directory\'}\n    changes = {k:{\'original\':original[k],\'external\':v} for k,v in config.items() if original[k] != v}\n    require(set(changes) <= allowed, \'Experimental parameter changed\')\n    return config, changes\n\n\ndef check_encoder(cache, report):\n    # Call exactly the same Transformers loader as the original CLI, then discard this untrained head.\n    os.environ[\'HF_HUB_OFFLINE\'] = \'1\'\n    os.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\n    from transformers import AutoModelForSequenceClassification\n    from nasn.marbert import seed_all\n    from nasn.preflight import LABEL2ID\n    seed_all(42,4)\n    model, info = AutoModelForSequenceClassification.from_pretrained(MODEL, revision=REVISION, cache_dir=cache,\n        local_files_only=True,num_labels=3,label2id=LABEL2ID,id2label={v:k for k,v in LABEL2ID.items()},\n        trust_remote_code=False,output_loading_info=True)\n    missing = [key for key in info[\'missing_keys\'] if not key.startswith((\'classifier.\',\'bert.pooler.\'))]\n    require(not missing and not info.get(\'error_msgs\') and not info.get(\'mismatched_keys\'),\n            f\'Incomplete pretrained encoder: {missing}\')\n    require(model.config.num_labels == 3 and model.config.label2id == LABEL2ID, \'Classification label mapping changed\')\n    save(report, {\'status\':\'encoder_verified\',\'revision\':REVISION,\'loading_info\':info,\n                  \'model_parameters\':sum(p.numel() for p in model.parameters()),\'training_performed\':False})\n    del model\n\n\ndef verify_completed_run(run):\n    run = Path(run)\n    required = REPORT_FILES - {\'near_duplicate_audit.csv\',\'near_duplicate_summary.csv\',\'manual_review_queue.csv\'}\n    require(all((run / name).is_file() for name in required), \'Completed-run artifacts are missing\')\n    status = json.loads((run / \'run_status.json\').read_text())\n    require(status[\'status\'] == \'completed\' and status[\'checkpoint_reload_verified\'] is True,\n            \'Run not completed or checkpoint reload not verified\')\n    require(status[\'test_evaluated\'] is False and status[\'model_revision\'] == REVISION,\n            \'Test evaluation or incorrect model revision detected\')\n    hashes = json.loads((run / \'checkpoint_hashes.json\').read_text())\n    require(\'model.safetensors\' in hashes, \'Trained model weight hash missing\')\n    checkpoint = (run / \'best_checkpoint\').resolve()\n    for name, digest in hashes.items():\n        file = (checkpoint / name).resolve()\n        require(file.is_relative_to(checkpoint) and file.is_file(), \'Invalid checkpoint hash entry\')\n        require(sha256(file) == digest, f\'Durable checkpoint hash mismatch: {name}\')\n    from nasn.metrics import calculate\n    import pandas as pd\n    predictions = pd.read_csv(run / \'validation_predictions.csv\',dtype={\'sarcasm\':str})\n    require(len(predictions) == 2509 and not predictions.source_row_id.duplicated().any(), \'Validation prediction coverage incorrect\')\n    config = json.loads((run / \'resolved_training_config.json\').read_text())\n    manifest = pd.read_csv(Path(config[\'split_directory\']) / \'validation_manifest.csv\',dtype=str)\n    preflight = json.loads((run / \'preflight.json\').read_text())\n    require(sha256(Path(config[\'split_directory\']) / \'validation_manifest.csv\') ==\n            preflight[\'manifest_sha256\'][\'validation_manifest.csv\'], \'Validation manifest changed\')\n    require(predictions.source_row_id.tolist() == manifest.source_row_id.tolist() and\n            predictions.gold_label.tolist() == manifest.sentiment.tolist(), \'Predictions do not match saved validation rows/labels\')\n    metrics, _, _ = calculate(predictions.gold_id,predictions.predicted_id)\n    saved = json.loads((run / \'validation_metrics.json\').read_text())\n    require(saved[\'status\'] == \'completed\', \'Validation not completed\')\n    for key in [\'accuracy\',\'macro_f1\',\'weighted_f1\']:\n        require(abs(saved[key] - metrics[key]) < 1e-12, f\'Saved validation metric differs from predictions: {key}\')\n    return status\n\n\ndef export_reports(run, target):\n    """Strict allowlist excludes weights, raw data, tokenizer dumps, and arbitrary logs."""\n    run, target = Path(run), Path(target)\n    require(not target.exists(), \'Lightweight report export directory already exists\')\n    target.mkdir(parents=True)\n    for name in sorted(REPORT_FILES):\n        path = run / name\n        if path.is_file():\n            require(path.stat().st_size <= 10 * 1024**2, f\'Report exceeds size limit: {name}\')\n            shutil.copyfile(path,target / name)\n    save(target / \'report_checksums.json\', {p.name:sha256(p) for p in sorted(target.iterdir()) if p.is_file()})\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\'command\',required=True)\n    gpu = sub.add_parser(\'gpu\'); gpu.add_argument(\'--report\',required=True)\n    snapshot = sub.add_parser(\'snapshot\')\n    for name in [\'cache\',\'lock\',\'report\']: snapshot.add_argument(\'--\'+name,required=True)\n    snapshot.add_argument(\'--local\')\n    encoder = sub.add_parser(\'encoder\')\n    encoder.add_argument(\'--cache\',required=True); encoder.add_argument(\'--report\',required=True)\n    verified = sub.add_parser(\'verify-run\'); verified.add_argument(\'--run\',required=True)\n    export = sub.add_parser(\'export\'); export.add_argument(\'--run\',required=True); export.add_argument(\'--target\',required=True)\n    args = parser.parse_args()\n    try:\n        if args.command == \'gpu\': gpu_report(args.report)\n        elif args.command == \'snapshot\': acquire_snapshot(args.cache,json.loads(Path(args.lock).read_text()),args.report,args.local)\n        elif args.command == \'encoder\': check_encoder(args.cache,args.report)\n        elif args.command == \'verify-run\': print(json.dumps(verify_completed_run(args.run),indent=2))\n        elif args.command == \'export\': export_reports(args.run,args.target)\n    except Exception as exc:\n        parser.exit(2,f\'ERROR: {type(exc).__name__}: {safe_error(exc)}\\n\')\n\nif __name__ == \'__main__\': main()\n'
HELPER_SHA256 = "2c3672386d517773f8b19fb586d3eecd9accf195f3595a5aa9022789de299918"
LOCK = json.loads('{\n  "repo_id": "UBC-NLP/MARBERT",\n  "revision": "88e1fa192dd723cf0b3563500aec46209762eb22",\n  "metadata_source": "https://huggingface.co/api/models/UBC-NLP/MARBERT/revision/88e1fa192dd723cf0b3563500aec46209762eb22?blobs=true",\n  "files": {\n    "config.json": {\n      "sha256": "cd2a25895e0e243e5475e241db2a61f5721b4b380ea175e9cafcbf4f0359b1f1",\n      "size": 701,\n      "git_blob_id": "67cea772d1e9b68a1c8ea80c85aa3efc7c362b39",\n      "verification_basis": "downloaded from immutable revision; Git blob identity and size verified"\n    },\n    "pytorch_model.bin": {\n      "sha256": "801862ace5fc9e6b135b19476dbda5c7de1e8673b5b9d55db08d79e2c30b9dca",\n      "size": 654186400,\n      "git_blob_id": "08fb7eefadf05e653c093f29f99dd877b94c1432",\n      "verification_basis": "official Hub LFS metadata; file bytes not downloaded in Task 03B development"\n    },\n    "special_tokens_map.json": {\n      "sha256": "303df45a03609e4ead04bc3dc1536d0ab19b5358db685b6f3da123d05ec200e3",\n      "size": 112,\n      "git_blob_id": "e7b0375001f109a6b8873d756ad4f7bbb15fbaa5",\n      "verification_basis": "downloaded from immutable revision; Git blob identity and size verified"\n    },\n    "tokenizer_config.json": {\n      "sha256": "16c98e576cebd77e6ea1c16ef01fc839af38606bbbb8dcc769b9a50f328ac54b",\n      "size": 376,\n      "git_blob_id": "9542287136eed6f2ddaa765fb79b40a32bb50876",\n      "verification_basis": "downloaded from immutable revision; Git blob identity and size verified"\n    },\n    "vocab.txt": {\n      "sha256": "4cbe17fdb035ba4afb5eb9889657b0f9895e2c6bd26574409cf9f8bdf75e2542",\n      "size": 1099714,\n      "git_blob_id": "d343b7b9b8071d9b7c43ba430feb53491b7fb5e9",\n      "verification_basis": "downloaded from immutable revision; Git blob identity and size verified"\n    }\n  }\n}\n')
HELPER = Path("/content/nasn_external_runtime.py")
HELPER.write_text(HELPER_SOURCE)
assert hashlib.sha256(HELPER.read_bytes()).hexdigest() == HELPER_SHA256
LOCK_PATH = RUN_ROOT / "model_snapshot_lock.json"
LOCK_PATH.write_text(json.dumps(LOCK,indent=2)+"\n")
(RUN_ROOT / "external_runtime_helper.py").write_text(HELPER_SOURCE)
spec = importlib.util.spec_from_file_location("nasn_external_runtime", HELPER)
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)


In [ ]:
# Verify CUDA build, GPU model/driver, total/free VRAM and a small actual CUDA matmul BEFORE model loading.
call([PYTHON,HELPER,"gpu","--report",RUN_ROOT / "gpu_runtime.json"],env=ENV,cwd=REPO)
GPU = json.loads((RUN_ROOT / "gpu_runtime.json").read_text())
print(json.dumps(GPU,indent=2))
ORIGINAL = json.loads((REPO / "configs/marbert_v1.json").read_text())
CONFIG, DEVIATIONS = runtime.make_config(ORIGINAL,GPU,CACHE,MICRO_BATCH_SIZE)
CONFIG_PATH = RUN_ROOT / "colab_training_config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2)+"\n")
(RUN_ROOT / "runtime_provenance.json").write_text(json.dumps({
    "reference_commit":REFERENCE,"helper_sha256":HELPER_SHA256,"cuda_index":CUDA_INDEX,
    "deviations":DEVIATIONS,"reason":"CUDA device/cache paths; microbatch by available VRAM, effective batch stays 16",
    "colab_kernel_python":sys.version,"managed_python":MANAGED_PYTHON,
    "run_id":RUN_ID,"durable_storage":str(RUN_ROOT),"training_executed":False},indent=2)+"\n")
print("Documented runtime changes:", json.dumps(DEVIATIONS,indent=2))


Free VRAM heuristic: microbatch 4 at ≥12 GiB, 2 at ≥8 GiB, otherwise 1 at ≥6 GiB. This is a conservative setup heuristic, not a capacity guarantee. On actual OOM, preserve the failed run, select a smaller microbatch and start a **new** run. Do not shorten epochs, change precision, truncate data, or tune using test results.


In [ ]:
# Fetch original pinned CSV bytes only. No split-generation command appears in this notebook.
call([PYTHON,"-m","arsarcasm_audit","fetch","--destination","data/raw"],cwd=REPO,env=ENV)
# Existing CLI checks all source/manifest hashes, 10037/2509/2/3000 counts and zero overlap.
call([PYTHON,"-m","nasn.marbert","preflight","--config",CONFIG_PATH,"--output",RUN_ROOT / "preflight"],cwd=REPO,env=ENV)
# Original reference-commit software tests; tiny random models are tests, never baseline results.
call([PYTHON,"-m","pytest","-q"],cwd=REPO,env=ENV)


## Exact pretrained snapshot (online or verified local)

If online download fails, the helper saves the host/error (signed query strings omitted) and stops. Do not repeat a failed command unchanged. For offline recovery, obtain the exact official snapshot on an accessible machine, verify it against this notebook's trusted lock, and upload it to Drive with `snapshot_revision.json` containing the exact repo/revision. Then set `LOCAL_SNAPSHOT`, rerun this cell and subsequent cells. A trained checkpoint, LFS pointer, mismatching revision, missing tokenizer, or altered weights fails verification.


In [ ]:
args = [PYTHON,HELPER,"snapshot","--cache",CACHE,"--lock",LOCK_PATH,"--report",RUN_ROOT / "snapshot_verification.json"]
if LOCAL_SNAPSHOT is not None:
    args += ["--local",LOCAL_SNAPSHOT]
call(args,cwd=REPO,env=ENV)
# Make the ORIGINAL model-name/revision CLI resolve exclusively to the verified cache.
# No changing config.model_name to a random/local model and no network fetch during training.
OFFLINE_ENV = {**ENV,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1"}
call([PYTHON,HELPER,"encoder","--cache",CACHE,"--report",RUN_ROOT / "encoder_verification.json"],cwd=REPO,env=OFFLINE_ENV)


## Run the unchanged documented baseline CLI

This is the only training invocation. Output/checkpoint directories are on Drive. The existing implementation saves improved checkpoints each epoch and reloads the best model **and tokenizer** for complete validation inference. It does not evaluate official test. A browser/runtime interruption is not successful completion and is not resumable by the original runner.


In [ ]:
assert not EXPERIMENT.exists(), "Never overwrite an experiment"
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip() == REFERENCE
assert not subprocess.check_output(["git","-C",str(REPO),"status","--porcelain","--untracked-files=no"],text=True).strip()
command = [str(PYTHON),"-m","nasn.marbert","train","--config",str(CONFIG_PATH),"--output",str(EXPERIMENT)]
(RUN_ROOT / "execution_command.json").write_text(json.dumps(command,indent=2)+"\n")
with (RUN_ROOT / "execution.log").open("w") as log:
    process = subprocess.Popen(command,cwd=REPO,env=OFFLINE_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        clean = runtime.safe_error(line)
        print(clean,end="")
        log.write(clean); log.flush()
    code = process.wait()
if code:
    (RUN_ROOT / "external_blocker.json").write_text(json.dumps({"status":"failed_or_blocked","exit_code":code,
        "run_status_file":str(EXPERIMENT / "run_status.json"),"training_completed":False},indent=2)+"\n")
    raise RuntimeError(f"Original CLI failed (exit {code}); inspect durable run_status.json and execution.log")


In [ ]:
# Require complete results and read back every checkpoint file from DURABLE storage.
call([PYTHON,HELPER,"verify-run","--run",EXPERIMENT],cwd=REPO,env=OFFLINE_ENV)
STATUS = json.loads((EXPERIMENT / "run_status.json").read_text())
assert STATUS["status"] == "completed" and STATUS["checkpoint_reload_verified"] and not STATUS["test_evaluated"]
provenance = json.loads((RUN_ROOT / "runtime_provenance.json").read_text())
provenance.update(training_executed=True,training_completed=True,run_status=STATUS)
(RUN_ROOT / "runtime_provenance.json").write_text(json.dumps(provenance,indent=2)+"\n")
print("Complete durable run verified:", EXPERIMENT)
print(json.dumps(json.loads((EXPERIMENT / "validation_metrics.json").read_text()),indent=2))


## Return lightweight reports to GitHub

The next cell exports an allowlist of CSV/JSON/Markdown experiment reports. It excludes `best_checkpoint/`, pretrained snapshots, raw CSVs, arbitrary logs and notebook outputs. Download the ZIP and review it before committing on a new report branch. Keep the full checkpoint and provenance in Drive; never put a token in this notebook. Exact Git commands are in `docs/task03b_colab.md` in the notebook's repository version.


In [ ]:
LIGHTWEIGHT = RUN_ROOT / "lightweight_reports"
call([PYTHON,HELPER,"export","--run",EXPERIMENT,"--target",LIGHTWEIGHT],cwd=REPO,env=OFFLINE_ENV)
# Include explicit helper/GPU/runtime provenance with the report bundle, not raw logs or files.
for name in ["gpu_runtime.json","runtime_provenance.json","snapshot_verification.json","encoder_verification.json","dependency_freeze.txt","dependency_check.txt"]:
    shutil.copyfile(RUN_ROOT / name,LIGHTWEIGHT / name)
(LIGHTWEIGHT / "report_checksums.json").write_text(json.dumps({
    file.name:runtime.sha256(file) for file in sorted(LIGHTWEIGHT.iterdir()) if file.name != "report_checksums.json"},indent=2)+"\n")
ZIP = shutil.make_archive(str(RUN_ROOT / (RUN_ID + "-reports")),"zip",LIGHTWEIGHT)
from google.colab import files
files.download(ZIP)
print("Trained weights stay in:", EXPERIMENT / "best_checkpoint")
